In [6]:
# Step 1: On terminal: nc -lk 9999
# Step 2: run this program
# Step 3: Enter some sentences on the terminal (on step 1)
# Step 4: Check program's console

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

# Create Spark session
spark = SparkSession.builder \
    .appName("StructuredStreamingExample") \
    .getOrCreate()

# Read streaming data from socket
lines = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# Transform: split words
words = lines.select(
    explode(split(lines.value, " ")).alias("word")
)

# Aggregation
wordCounts = words.groupBy("word").count()

# Write output to console
query = wordCounts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = 449bc580-ec96-4d1a-ac0e-2c8a61e6209b, runId = 7039fb15-fc3c-4dca-86e5-af31b37c2d30] terminated with exception: Connection refused